# Advanced: Multi-Strategy Categorical Analysis with PAMOLA.CORE

**Goal**: Master all 3 categorical profiling strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Standard profiling with balanced settings
- **Strategy 2**: Deep profiling with high-resolution analysis
- **Strategy 3**: Quality-focused profiling with anomaly detection
- Compare analysis depth and detection effectiveness
- Cross-field distribution analysis
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of data profiling concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 02_categorical_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.categorical import (
        CategoricalOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 5 rows)
- Column statistics (3 categorical fields with varying complexity)

**Dataset features:**
- Multiple categorical fields (status, category, region)
- Intentional data quality issues (typos, rare values, single chars)
- Varying distribution patterns (concentrated vs distributed)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'categorical_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Status field - concentrated distribution with typos
    statuses = (
        ['Active', 'Pending', 'Completed', 'Cancelled'] * 240
        + ['Activ', 'Actve', 'P', 'C']
        + ['Unknown'] * 36
    )

    # Category field - well-distributed with some anomalies
    categories = (
        ['Electronics', 'Clothing', 'Books', 'Home',
        'Sports', 'Beauty', 'Toys', 'Automotive'] * 120
        + ['Eletronics', 'Electrncs', 'E', 'B', '123'] * 2
        + ['Misc', 'Other'] * 15
    )

    # Region field - moderately distributed with quality issues
    regions = (
        ['North America', 'Europe', 'Asia', 'South America',
        'Africa', 'Oceania'] * 160
        + ['NA', 'EU', 'AS', 'N'] * 5
        + ['Eurpe', 'Asi'] * 10
    )
    
    df = pd.DataFrame({
        'record_id': [f'R{i:04d}' for i in range(1, n_records + 1)],
        'status': statuses,
        'category': categories,
        'region': regions,
        'priority': np.random.choice(['High', 'Medium', 'Low'], n_records, p=[0.2, 0.5, 0.3]),
        'value': np.random.randint(10, 1000, n_records),
        'quantity': np.random.randint(1, 100, n_records),
        'verified': np.random.choice(['Yes', 'No'], n_records, p=[0.8, 0.2])
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Dataset includes intentional quality issues for testing all strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field assignments for each strategy
   - `strategy1_field`: For standard profiling (default: "status")
   - `strategy2_field`: For deep profiling (default: "category")
   - `strategy3_field`: For quality profiling (default: "region")
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

**This setup will be reused for all 3 strategies**

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "status",      # Standard profiling
    "strategy2_field": "category",    # Deep profiling
    "strategy3_field": "region"       # Quality-focused profiling
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="categorical_adv_001",
    task_type="multi_strategy_profiling",
    description="Multi-strategy categorical field profiling",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Standard Profiling

**How to use:**
- Balanced settings for general-purpose analysis
- Run to execute standard profiling strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → analysis → metrics → viz → save
- Completion time and status
- Analysis summary (unique values, entropy)

**Key parameters:**
- `top_n=15`: Show top 15 most frequent values
- `min_frequency=1`: Include all values in dictionary
- `analyze_anomalies=True`: Basic anomaly detection

**Note:** Good baseline for comparing other strategies

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: STANDARD PROFILING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Standard",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = CategoricalOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    top_n=15,
    min_frequency=1,
    
    # Analysis configuration
    profile_type='categorical',
    analyze_anomalies=True,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_standard',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Display key metrics
if result_s1.metrics:
    print(f"\n📊 Analysis Summary:")
    print(f"   Unique values: {result_s1.metrics.get('unique_values', 0)}")
    print(f"   Entropy: {result_s1.metrics.get('entropy', 0):.4f}")
    print(f"   Anomalies found: {result_s1.metrics.get('anomalies_count', 0)}")

## STRATEGY 2: Deep Profiling

**How to use:**
- High-resolution analysis with more details
- Run to execute deep profiling strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → analysis → metrics → viz → save
- Completion time and status
- Enhanced analysis with more top values

**Key parameters:**
- `top_n=30`: Show top 30 values (2x standard)
- `min_frequency=1`: Comprehensive dictionary
- `analyze_anomalies=True`: Full anomaly detection

**Note:** Best for fields with many unique values requiring detailed analysis

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: DEEP PROFILING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6, 
    description="Strategy 2: Deep", 
    unit="steps"
)

operation_s2 = CategoricalOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    top_n=30,  # Double the standard for deeper insights
    min_frequency=1,
    profile_type='categorical',
    analyze_anomalies=True,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_deep',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

if result_s2.metrics:
    print(f"\n📊 Analysis Summary:")
    print(f"   Unique values: {result_s2.metrics.get('unique_values', 0)}")
    print(f"   Entropy: {result_s2.metrics.get('entropy', 0):.4f}")
    print(f"   Anomalies found: {result_s2.metrics.get('anomalies_count', 0)}")

## STRATEGY 3: Quality-Focused Profiling

**How to use:**
- Emphasis on data quality and anomaly detection
- Run to execute quality-focused strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → analysis → metrics → viz → save
- Completion time and status
- Enhanced anomaly detection results

**Key parameters:**
- `top_n=20`: Moderate top values display
- `min_frequency=2`: Focus on values appearing 2+ times
- `analyze_anomalies=True`: Maximum anomaly sensitivity

**Note:** Optimal for detecting typos, single characters, and rare values

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: QUALITY-FOCUSED PROFILING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6, 
    description="Strategy 3: Quality", 
    unit="steps"
)

operation_s3 = CategoricalOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    top_n=20,
    min_frequency=2,  # Higher threshold for quality focus
    profile_type='categorical',
    analyze_anomalies=True,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_quality',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

if result_s3.metrics:
    print(f"\n📊 Analysis Summary:")
    print(f"   Unique values: {result_s3.metrics.get('unique_values', 0)}")
    print(f"   Entropy: {result_s3.metrics.get('entropy', 0):.4f}")
    print(f"   Anomalies found: {result_s3.metrics.get('anomalies_count', 0)}")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Field complexity comparison (unique values, entropy)
- Anomaly detection effectiveness

**Note:** Compare strategies to choose best approach for your data characteristics

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Standard): {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Deep):     {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Quality):  {elapsed_s3:6.2f}s")
print(f"  Total:                 {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Field Complexity:")
print(f"  Strategy 1: {result_s1.metrics.get('unique_values', 0):3d} unique, "
      f"entropy={result_s1.metrics.get('entropy', 0):.4f}")
print(f"  Strategy 2: {result_s2.metrics.get('unique_values', 0):3d} unique, "
      f"entropy={result_s2.metrics.get('entropy', 0):.4f}")
print(f"  Strategy 3: {result_s3.metrics.get('unique_values', 0):3d} unique, "
      f"entropy={result_s3.metrics.get('entropy', 0):.4f}")

print(f"\n🔍 Anomaly Detection:")
print(f"  Strategy 1: {result_s1.metrics.get('anomalies_count', 0)} anomalies")
print(f"  Strategy 2: {result_s2.metrics.get('anomalies_count', 0)} anomalies")
print(f"  Strategy 3: {result_s3.metrics.get('anomalies_count', 0)} anomalies")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Total records, nulls, unique values, entropy
2. **Statistics JSON**: Distribution type, top values, diversity metrics
3. **Value Dictionary**: Frequency table (CSV)
4. **Anomalies**: Detected quality issues (CSV)
5. **Visualizations**: Distribution charts

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_standard', 'Strategy 1: Standard Profiling'),
    ('strategy2_deep', 'Strategy 2: Deep Profiling'),
    ('strategy3_quality', 'Strategy 3: Quality-Focused')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                print(f"   Total records: {metrics.get('total_records', 0)}")
                print(f"   Null count: {metrics.get('null_count', 0)}")
                print(f"   Unique values: {metrics.get('unique_values', 0)}")
                print(f"   Entropy: {metrics.get('entropy', 0):.4f}")
                print(f"   Distribution: {metrics.get('distribution_type', 'N/A')}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Statistics (NEWEST)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        stats_files = sorted(
            list(output_dir.glob('*_stats_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if stats_files:
            print(f"\n📄 STATISTICS: {stats_files[0].name}")
            try:
                with open(stats_files[0], 'r') as f:
                    stats = json.load(f)
                print(f"   Field: {stats.get('field_name', 'N/A')}")
                print(f"   Cardinality ratio: {stats.get('cardinality_ratio', 0):.4f}")
                print(f"   Top values count: {len(stats.get('top_values', {}))}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 3. Dictionary (NEWEST)
    dict_dir = strategy_dir / 'dictionaries'
    if dict_dir.exists():
        dict_files = sorted(
            list(dict_dir.glob('*_dictionary_*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if dict_files:
            print(f"\n📄 DICTIONARY: {dict_files[0].name}")
            try:
                dict_df = pd.read_csv(dict_files[0])
                print(f"   Total values: {len(dict_df)}")
                if len(dict_df) > 0:
                    print(f"   Most frequent: {dict_df.iloc[0].to_dict()}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 4. Anomalies (NEWEST)
    if dict_dir and dict_dir.exists():
        anomaly_files = sorted(
            list(dict_dir.glob('*_anomalies_*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if anomaly_files:
            print(f"\n📄 ANOMALIES: {anomaly_files[0].name}")
            try:
                anomaly_df = pd.read_csv(anomaly_files[0])
                print(f"   Total anomalies: {len(anomaly_df)}")
                if 'anomaly_type' in anomaly_df.columns:
                    type_counts = anomaly_df['anomaly_type'].value_counts()
                    for atype, count in type_counts.items():
                        print(f"   - {atype}: {count}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 5. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Cross-Field Analysis

**How to use:**
- Run AFTER all strategies complete
- Compare distribution patterns across fields

**What you'll see:**
- Distribution type comparison (concentrated vs distributed)
- Entropy comparison (diversity measure)
- Cardinality ratios (uniqueness measure)
- Data quality scores

**Interpretation:**
- Higher entropy = more diverse distribution
- Lower concentration = better distribution
- More anomalies = potential quality issues

**Note:** Helps identify fields needing attention or cleanup

In [ ]:
print("\n" + "=" * 80)
print("📊 CROSS-FIELD ANALYSIS")
print("=" * 80 + "\n")

# Collect statistics from all strategies
field_stats = []

for strategy_id, (dir_name, title) in enumerate(strategy_dirs, 1):
    strategy_dir = task_dir / dir_name
    output_dir = strategy_dir / 'output'
    
    if output_dir.exists():
        stats_files = sorted(
            list(output_dir.glob('*_stats_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if stats_files:
            try:
                with open(stats_files[0], 'r') as f:
                    stats = json.load(f)
                    field_stats.append({
                        'strategy': f'Strategy {strategy_id}',
                        'field': stats.get('field_name', 'N/A'),
                        'unique': stats.get('unique_values', 0),
                        'entropy': stats.get('entropy', 0),
                        'cardinality': stats.get('cardinality_ratio', 0),
                        'distribution': stats.get('distribution_type', 'N/A'),
                        'concentration': stats.get('concentration', 0)
                    })
            except Exception as e:
                print(f"Warning: Could not load stats for {title}: {e}")

if field_stats:
    # Create comparison DataFrame
    comparison_df = pd.DataFrame(field_stats)
    
    print("📋 DISTRIBUTION CHARACTERISTICS:")
    print("=" * 80)
    print(comparison_df.to_string(index=False))
    
    print(f"\n\n📈 DIVERSITY RANKING (by entropy):")
    print("=" * 80)
    sorted_by_entropy = comparison_df.sort_values('entropy', ascending=False)
    for i, row in sorted_by_entropy.iterrows():
        print(f"  {row['strategy']}: {row['field']:15s} (entropy={row['entropy']:.4f})")
    
    print(f"\n\n🎯 DATA QUALITY ASSESSMENT:")
    print("=" * 80)
    for stat in field_stats:
        quality_score = 100  # Start with perfect score
        
        # Deduct points for concentration
        if stat['concentration'] > 0.8:
            quality_score -= 20
        elif stat['concentration'] > 0.6:
            quality_score -= 10
        
        # Deduct points for low entropy (less diversity)
        if stat['entropy'] < 1.0:
            quality_score -= 15
        elif stat['entropy'] < 2.0:
            quality_score -= 5
        
        print(f"  {stat['strategy']}: {stat['field']:15s} = {quality_score}/100")
        
        if quality_score < 70:
            print(f"      ⚠️  Potential quality issues detected")
        elif quality_score < 85:
            print(f"      ℹ️  Moderate distribution concentration")
        else:
            print(f"      ✅ Good distribution characteristics")
else:
    print("⚠️  No field statistics available for comparison")

## Step 7: Export Analysis Results

**How to use:**
- Run AFTER all strategies complete
- Exports analysis results for each strategy

**What you'll see (per strategy):**
- Export path confirmation
- File list (statistics, dictionary, anomalies)
- File sizes and row counts

**Output structure:**
```
advanced_output/
├── strategy1_standard/
│   ├── output/          # Statistics JSON
│   ├── dictionaries/    # Dictionary & anomalies CSV
│   └── visualizations/  # Charts PNG
├── strategy2_deep/
│   └── ...
└── strategy3_quality/
    └── ...
```

**Note:** All artifacts preserved for external analysis or reporting

In [ ]:
print("=" * 80)
print("📦 EXPORTING ANALYSIS RESULTS FROM ALL STRATEGIES")
print("=" * 80)

print(f"\n📂 Export base directory: {task_dir}\n")

# Review exported files for each strategy
for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    
    if not strategy_dir.exists():
        print(f"\n⚠️  {title}: Directory not found")
        continue
    
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # Count files in each subdirectory
    for subdir_name in ['output', 'dictionaries', 'visualizations']:
        subdir = strategy_dir / subdir_name
        if subdir.exists():
            files = list(subdir.glob('*'))
            print(f"\n📁 {subdir_name}/ ({len(files)} files):")
            
            for f in files:
                size_kb = f.stat().st_size / 1024
                print(f"   • {f.name} ({size_kb:.1f} KB)")
                
                # For CSV files, show row count
                if f.suffix == '.csv':
                    try:
                        temp_df = pd.read_csv(f)
                        print(f"     Rows: {len(temp_df):,}")
                    except:
                        pass

print("\n\n" + "=" * 80)
print("✅ EXPORT SUMMARY")
print("=" * 80)
print(f"\n📂 All files saved to: {task_dir}")

if 'elapsed_s1' in locals() and 'elapsed_s2' in locals() and 'elapsed_s3' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 3 profiling strategies implemented and compared
- ✅ Multi-field analysis with varying complexity
- ✅ Distribution characteristics analyzed
- ✅ Anomaly detection effectiveness measured
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Standard profiling provides balanced insights for most use cases
- Deep profiling reveals detailed patterns in complex fields
- Quality-focused profiling excels at detecting data issues
- Entropy and concentration metrics indicate distribution health
- Anomaly detection identifies typos, rare values, and quality problems

**Strategy recommendations:**
- **Use Standard** when: General profiling, balanced depth needed
- **Use Deep** when: High cardinality fields, detailed analysis required
- **Use Quality** when: Data quality assessment, anomaly detection priority

**Next steps:**
- Apply strategies to your production datasets
- Tune `top_n` and `min_frequency` for your data characteristics
- Integrate profiling into data quality pipelines
- Use anomaly reports for data cleaning initiatives

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)